# Stage 1.5: Create Seasonal Subsets

This notebook splits the processed `merged_gdf.feather` into **8 subset datasets** per parameter,
one for each combination of:
- **Season**: Spring, Summer, Autumn, Winter
- **Depth layer**: Surface (`saisuipoin == 1`), Bottom (`saisuipoin == 2`)

For each subset it computes the **yearly mean per station** and saves a
`yearly_{param}_df.csv` inside:
```
data/processed/seasonal/{season_lower}_{layer_lower}/
```

> **Note**: The station-to-station distance matrix (`dist_matrix_km.csv`) depends
> only on station geography (not on season or layer) and is therefore **shared**.
> It is produced once by `08_Decorrelation_Analysis_v3.ipynb` and lives at
> `data/processed/dist_matrix_km.csv`.

In [1]:
import pandas as pd
import sys
from pathlib import Path

# Make project scripts importable
project_root = Path("../../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils.paths import ProjectPaths
from scripts.data_loader import load_and_process_data

print("Project root:", project_root)

Project root: /home/reyha/SIS_Analysis_WSL


In [2]:
# Parameters (injectable via papermill / run_pipeline_step.py)
min_periods = 15  # stations must have >= this many years of data to be included
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
layers  = {'surface': 1, 'bottom': 2}

# Representative month per season (based on sampling_months_check.ipynb analysis)
# Only observations from these months are used to represent each season.
representative_months = {
    'Spring': 5,   # May
    'Summer': 7,   # July
    'Autumn': 10,  # October
    'Winter': 1    # January
}
# Parameters to process
parameters_to_process = ['do_', 'temp', 'wtemp', 'salt']


## 1. Load and filter the merged dataset

(edit) Add a filtering step to select between Seto Inland Sea, Tokyo Bay, or Ise-Mikawa Bay.

In [3]:
# Load the cleaned, merged dataset
merged_gdf = load_and_process_data(force_reload=False)

# Add Regional Filtering (Add this right after loading)
try:
    import yaml
    with open('../../params.yaml', 'r') as f:
        params = yaml.safe_load(f)
    TARGET_REGION = params['mapping']['target_region']
    REGION_COLUMN = params['mapping']['region_column']
    
    # Filter the dataset
    initial_count = len(merged_gdf)
    merged_gdf = merged_gdf[merged_gdf[REGION_COLUMN] == TARGET_REGION].copy()
    print(f"Filtered for {TARGET_REGION}: {initial_count} -> {len(merged_gdf)} rows")
except Exception as e:
    print(f"Regional filtering skipped or failed: {e}")

Loading processed data from /home/reyha/SIS_Analysis_WSL/data/processed/merged_gdf.feather...
Filtered for Seto Inland Sea: 62738 -> 42326 rows


## 2. Create one subset directory per season × layer combination

For each subset and parameter:
1. Filter rows by season, `saisuipoin`, **and the representative month** for that season.
2. Drop rows where the parameter is NaN.
3. Aggregate to yearly mean parameter per station (`zettaicode`).
4. Drop stations with fewer than `min_periods` non-NaN years.
5. Save as `yearly_{param}_df.csv` in the appropriate subdirectory.

> **Note**: Each season uses a single representative month (May/Jul/Oct/Jan) 
> as determined by the sampling density analysis (`sampling_months_check.ipynb`).

> ⚠️ **CAUTION - Missing Data Years**: Because we strictly enforce sampling in the single `representative_month` for each season (e.g., Month 7 for Summer), any year where sampling randomly occurred slightly earlier or later (e.g., June or August instead of July) will be entirely dropped from the resulting `yearly_{param}_df.csv` matrix. 
> *Example: 1981 and 1983 are missing from the Summer matrices because the stations were not sampled in July during those specific years.*


In [4]:
results_summary = []

for season in seasons:
    for layer_name, layer_code in layers.items():
        rep_month = representative_months[season]
        
        # --- 1. Filter by season, layer, AND representative month ---
        base_subset = merged_gdf[
            (merged_gdf['season'] == season) &
            (merged_gdf['saisuipoin'] == layer_code) &
            (merged_gdf['surveymont'] == rep_month)
        ].copy()
        
        out_dir = ProjectPaths.ensure_subset_dir(season, layer_name)
        
        for param in parameters_to_process:
            subset = base_subset.copy()
            subset[param] = pd.to_numeric(subset[param], errors='coerce')
            subset = subset.dropna(subset=[param]).copy()
            
            # --- 2. Yearly mean parameter per station ---
            yearly_df = (
                subset
                .groupby(['zettaicode', 'year'])[param]
                .mean()
                .unstack('year')
            )

            # --- 3. Drop stations below min_periods threshold ---
            yearly_df = yearly_df[yearly_df.notna().sum(axis=1) >= min_periods]
            
            # Ensure the station index is zero-padded 8-digit strings
            yearly_df.index = yearly_df.index.astype(str).str.zfill(8)
            
            # --- 4. Save ---
            param_filename = param.rstrip('_')
            out_path = out_dir / f'yearly_{param_filename}_df.csv'
            yearly_df.to_csv(out_path, index=True)
            
            results_summary.append({
                'season': season,
                'layer': layer_name,
                'saisuipoin': layer_code,
                'param': param,
                'rep_month': rep_month,
                'n_stations': yearly_df.shape[0],
                'n_years': yearly_df.shape[1],
                'output': str(out_path)
            })
            print(f"[{season:6s} / {layer_name:7s} | {param:5s}] month={rep_month:2d} stations={yearly_df.shape[0]:3d} years={yearly_df.shape[1]} -> {out_path}")

print("\nAll subsets saved.")

[Spring / surface | do_  ] month= 5 stations=124 years=43 -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_surface/yearly_do_df.csv
[Spring / surface | temp ] month= 5 stations=124 years=43 -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_surface/yearly_temp_df.csv
[Spring / surface | wtemp] month= 5 stations=124 years=43 -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_surface/yearly_wtemp_df.csv
[Spring / surface | salt ] month= 5 stations=124 years=43 -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_surface/yearly_salt_df.csv
[Spring / bottom  | do_  ] month= 5 stations=124 years=43 -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_bottom/yearly_do_df.csv
[Spring / bottom  | temp ] month= 5 stations=124 years=38 -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spring_bottom/yearly_temp_df.csv
[Spring / bottom  | wtemp] month= 5 stations=124 years=43 -> /home/reyha/SIS_Analysis_WSL/data/processed/seasonal/spr

## 3. Summary

In [5]:
summary_df = pd.DataFrame(results_summary)
display(summary_df)

,season,layer,saisuipoin,param,rep_month,n_stations,n_years,output
0,Spring,surface,1,do_,5,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
1,Spring,surface,1,temp,5,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
2,Spring,surface,1,wtemp,5,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
3,Spring,surface,1,salt,5,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
4,Spring,bottom,2,do_,5,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
5,Spring,bottom,2,temp,5,124,38,/home/reyha/SIS_Analysis_WSL/data/processed/se...
6,Spring,bottom,2,wtemp,5,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
7,Spring,bottom,2,salt,5,124,43,/home/reyha/SIS_Analysis_WSL/data/processed/se...
8,Summer,surface,1,do_,7,124,41,/home/reyha/SIS_Analysis_WSL/data/processed/se...
9,Summer,surface,1,temp,7,124,41,/home/reyha/SIS_Analysis_WSL/data/processed/se...
